# Task 2 — ResNet-18 from Scratch on GTSRB
**Name:**MD Ayaan | **Roll No:** 25155053|

---

implementing ResNet-18 from the original paper:

> He et al., "Deep Residual Learning for Image Recognition", 2015

the whole point of ResNet is solving the vanishing gradient problem in deep networks.
the key idea: instead of learning H(x) directly, learn the residual F(x) = H(x) - x.
then the output is F(x) + x.

why this helps with gradients: during backprop,
d(F(x)+x)/dx = dF/dx + 1

that +1 means even if dF/dx becomes near-zero (vanishes), there's always a constant gradient flowing through. without skip connections in a 50+ layer network, early layers basically stop learning.

i also watched the andrew ng deep learning lecture on residual networks — he explains the gradient flow really clearly, helped me understand why this works.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


import numpy as np
import matplotlib.pyplot as plt
import os

from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

NUM_CLASSES = 43

device: cuda


## getting the dataset from kaggle

In [5]:
from pathlib import Path
import kagglehub

# Download latest version
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")

print("Path to dataset files:", path)


DATA_PATH = path

TRAIN_PATH = os.path.join(DATA_PATH, 'Train')


classes = sorted(os.listdir(TRAIN_PATH))
print('number of classes:', len(classes))

Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.
Path to dataset files: /kaggle/input/gtsrb-german-traffic-sign
number of classes: 43


## Dataset

using 224×224 because that's what the ResNet paper uses. also using ImageNet mean/std normalization — even though we're training from scratch this still helps because the values are in a similar range to what the weight init expects.

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),

    # imagenet mean/std — standard for resnet even when training from scratch
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])



val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [7]:


class GTSRBDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label_idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if fname.endswith('.png') or fname.endswith('.ppm'):
                    self.samples.append((os.path.join(class_dir, fname), int(label_idx)))

        print(f'loaded: {len(self.samples)} samples')




    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [8]:
# load + split manually
full_dataset = GTSRBDataset(TRAIN_PATH, transform=None)
all_samples = full_dataset.samples

np.random.seed(42)
np.random.shuffle(all_samples)

n_train = int(0.8 * len(all_samples))
train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:]

# two separate dataset objects with diff transforms
class SplitDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform


    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = SplitDataset(train_samples, train_transform)
val_dataset   = SplitDataset(val_samples,   val_transform)

# num_workers=4, pin_memory=True for speed on 224x224 images
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print('train batches:', len(train_loader))
print('val batches:',   len(val_loader))

loaded: 39209 samples
train batches: 491
val batches: 123


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## ResNet-18 Architecture

### BasicBlock

each block has: conv3x3 → BN → ReLU → conv3x3 → BN → (add shortcut) → ReLU

important: relu comes AFTER the addition, not inside the second conv. i got this wrong the first time and the training was worse.

when the dimensions change between blocks (different channels or stride=2), i can't just add x directly because the shapes don't match. so i use a 1×1 conv (projection shortcut) on the skip path to resize x to match.

mathematically:
- same dims: output = ReLU(F(x) + x)
- diff dims: output = ReLU(F(x) + W_s·x) where W_s is a 1×1 conv

In [9]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)

        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)


        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)


        # default shortcut is just identity
        self.shortcut = nn.Identity()



        # if dims change we need a projection shortcut
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)


        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))  # no relu here yet

        out = out + identity  # the actual residual connection
        out = self.relu(out)  # relu AFTER the add

        return out

In [10]:

class ResNet18(nn.Module):
    def __init__(self, num_classes=43):
        super(ResNet18, self).__init__()

        # initial layer: 7x7 conv, 64 filters, stride 2 (paper table 1)
        self.conv1   = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)


        self.layer1 = self._make_stage(64,  64,  num_blocks=2, stride=1)
        self.layer2 = self._make_stage(64,  128, num_blocks=2, stride=2)
        self.layer3 = self._make_stage(128, 256, num_blocks=2, stride=2)
        self.layer4 = self._make_stage(256, 512, num_blocks=2, stride=2)


        # global avg pool → flatten → fc



        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))  # output always 1x1 regardless of input
        self.fc      = nn.Linear(512, num_classes)

        self._init_weights()




    def _make_stage(self, in_channels, out_channels, num_blocks, stride):
        layers = [BasicBlock(in_channels, out_channels, stride=stride)]
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)





    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)






    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [11]:

model = ResNet18(num_classes=43).to(device)

dummy = torch.zeros(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print('output shape:', out.shape)  # [2, 43]

total_params = sum(p.numel() for p in model.parameters())
print(f'total params: {total_params:,}')



output shape: torch.Size([2, 43])
total params: 11,198,571


## Training

using SGD with momentum like the original paper (they use lr=0.1 but i start at 0.01 since our dataset is smaller and i didn't want to overshoot).

CosineAnnealingLR — smoothly reduces lr over training instead of step drops. heard this works better than StepLR for ResNet style architectures.

In [ ]:
model = ResNet18(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()

# optimizer = optim.Adam(model.parameters(), lr=1e-3)  # tried this first, SGD was better
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

NUM_EPOCHS = 30

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # TODO: add val loop here
    # ran out of time to finish this part properly

    if (epoch+1) % 5 == 0:
        print(f'epoch {epoch+1} | loss: {train_loss:.3f} | acc: {train_acc:.3f}')

# TODO: save best model logic
# torch.save(model.state_dict(), 'resnet18_best.pth')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# loss curve — only have train loss rn since val loop isnt done
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='train loss', color='steelblue')
# plt.plot(val_losses, label='val loss', color='orange')  # TODO
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('training loss (val pending)')
plt.legend()
plt.grid(True)
plt.savefig('resnet18_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## TODO — Pending (ran out of time)

ok so i ran out of time to finish this task completely.
i got the architecture built and training loop running but the val loop, confusion matrix and per-class accuracy are left.

what's done:
- dataset loading and transforms ✓
- BasicBlock with projection shortcut ✓
- full ResNet18 architecture ✓  
- training loop (train side only) ✓

what's left:
- validation loop inside training
- save best model logic
- confusion matrix
- sample predictions plot
- per-class accuracy

will finish this after submitting if sir allows. the architecture part is the main thing anyway i think.